In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folder
!mkdir -p /content/drive/MyDrive/dementia_risk_project/data/raw
!mkdir -p /content/drive/MyDrive/dementia_risk_project/data/processed
!mkdir -p /content/drive/MyDrive/dementia_risk_project/src
!mkdir -p /content/drive/MyDrive/dementia_risk_project/results

# Symlink for easy access
!ln -sf /content/drive/MyDrive/dementia_risk_project /content/dementia_risk_project
%cd /content/dementia_risk_project

Mounted at /content/drive
/content/drive/MyDrive/dementia_risk_project


In [ ]:
!pip install pandas numpy tqdm scikit-learn torch transformers datasets sentence-transformers xgboost matplotlib seaborn statsmodels -q
!pip uninstall -y sympy
!pip install sympy==1.12

In [ ]:
# Download the dataset (this may take 5-10 minutes)
print("Downloading PMC-Patients dataset...")
snapshot_download(repo_id="zhengyun21/PMC-Patients",
                  local_dir="/content/dementia_risk_project/data/raw/pmc_patients",
                  repo_type="dataset")

print("Download complete!")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Download complete!


In [ ]:


print("Loading PMC-Patients data...")

# Load the main JSON file
with open("/content/dementia_risk_project/data/raw/pmc_patients/PMC-Patients-V2.json", "r") as f:
    all_patients = json.load(f)

print(f"Loaded {len(all_patients)} total patients")

# Define keywords
DEMENTIA_KEYWORDS = [
    "dementia", "Alzheimer", "Alzheimer's", "vascular dementia",
    "Lewy body", "frontotemporal dementia", "cognitive decline",
    "memory loss", "mild cognitive impairment", "MCI"
]

CONTROL_KEYWORDS = [
    "hip fracture", "hip replacement", "knee replacement",
    "cataract", "macular degeneration", "osteoporosis",
    "prostate cancer", "hernia repair", "pneumonia",
    "urinary tract infection", "atrial fibrillation"
]

def contains_keywords(text, keywords):
    if not text:
        return False
    text_lower = text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

# Filter dementia cases
dementia_cases = [p for p in all_patients if contains_keywords(p.get("patient", ""), DEMENTIA_KEYWORDS)]
print(f"Dementia cases: {len(dementia_cases)}")

# Filter control candidates
control_candidates = [
    p for p in all_patients
    if contains_keywords(p.get("patient", ""), CONTROL_KEYWORDS)
    and not contains_keywords(p.get("patient", ""), DEMENTIA_KEYWORDS)
]
print(f"Control candidates: {len(control_candidates)}")

# Balance dataset
min_size = min(len(dementia_cases), len(control_candidates), 350)  # Limit to 2000 each for Colab
dementia_cases = dementia_cases[:min_size]
control_candidates = control_candidates[:min_size]

# Combine and label
for p in dementia_cases:
    p["label_dementia"] = 1
for p in control_candidates:
    p["label_dementia"] = 0

all_filtered = dementia_cases + control_candidates
random.shuffle(all_filtered)

print(f"Final dataset: {len(all_filtered)} patients ({len(dementia_cases)} dementia, {len(control_candidates)} control)")

# Save
with open("data/processed/filtered_patients.json", "w") as f:
    json.dump(all_filtered, f)

print("Saved to data/processed/filtered_patients.json")

Loading PMC-Patients data...
Loaded 250294 total patients
Dementia cases: 8082
Control candidates: 29960
Final dataset: 700 patients (350 dementia, 350 control)
Saved to data/processed/filtered_patients.json


Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 66.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires sympy>=1.13.3, but you have sympy 1.12 which is incompatible.


In [ ]:
# Load filtered patients
with open("data/processed/filtered_patients.json", "r") as f:
    patients = json.load(f)

print(f"Processing {len(patients)} patients...")
model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded. Device: {model.device}")

# Rest of your code stays the same...
print(f"VRAM usage: ~{torch.cuda.memory_allocated()/1e9:.1f} GB")



Processing 700 patients...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded. Device: cuda:0
VRAM usage: ~5.2 GB


In [ ]:
PROMPT_TEMPLATE = """Extract structured information from this clinical summary. Output ONLY valid JSON.

Clinical Summary: {summary}

Fields:
- age_years (integer)
- sex ("M"/"F")
- hypertension (true/false)
- brain_injury (true/false)
- diabetes (true/false)
- depression (true/false)
- smoking (true/false)
- obesity (true/false)
- afib (true/false)
- stroke (true/false)
- alcohol (true/false)

JSON:"""

class RegexExtractor:
    """Efficient regex extractor with pre-compiled patterns"""

    def __init__(self):
        # Age patterns
        self.age_patterns = [
            re.compile(r'(\d+)-year-old', re.I),
            re.compile(r'(\d+)\s+years?\s+old', re.I),
            re.compile(r'age\s+(\d+)', re.I),
            re.compile(r'aged\s+(\d+)', re.I),
            re.compile(r'(\d+)\s+year\s+old', re.I)
        ]

        # Sex patterns
        self.female_pattern = re.compile(r'\b(female|woman|girl|lady|f)\b', re.I)
        self.male_pattern = re.compile(r'\b(male|man|boy|m)\b', re.I)

        # Risk factor patterns
        self.patterns = {
            'hypertension': re.compile(r'\b(hypertension|high bp|high blood pressure|htn)\b', re.I),
            # 'hyperlipidemia': re.compile(r'\b(hyperlipidemia|high cholesterol|high lipids|dyslipidemia)\b', re.I),
            'brain_injury': re.compile(r'\b(traumatic brain injury|tbi|head injury|brain trauma)\b', re.I),
            'diabetes': re.compile(r'\b(diabetes|type 2 diabetes|t2dm|dm|diabetic)\b', re.I),
            'depression': re.compile(r'\b(depression|depressed|major depressive disorder|mdd|clinical depression)\b', re.I),
            'smoking': re.compile(r'\b(smoking|smoker|current smoker|former smoker|tobacco)\b', re.I),
            'obesity': re.compile(r'\b(obesity|obese|bmi\s*>\s*30|bmi\s*≥\s*30)\b', re.I),
            'afib': re.compile(r'\b(atrial fibrillation|afib|a-fib)\b', re.I),
            'stroke': re.compile(r'\b(stroke|cva|cerebrovascular accident|tia)\b', re.I),
            'alcohol': re.compile(r'\b(alcohol|alcoholism|alcohol abuse|heavy drinking)\b', re.I),
        }

    def extract(self, summary):
        summary_lower = summary.lower()

        # Age
        age_years = None
        for pattern in self.age_patterns:
            match = pattern.search(summary_lower)
            if match:
                age_years = int(match.group(1))
                break

        # Sex
        sex = None
        if self.female_pattern.search(summary_lower):
            sex = 'F'
        elif self.male_pattern.search(summary_lower):
            sex = 'M'

        # Risk factors
        results = {'age_years': age_years, 'sex': sex}
        for key, pattern in self.patterns.items():
            results[key] = bool(pattern.search(summary_lower))

        return results


# Compare regex vs LLM vs ground truth

# Define your extraction schema
class DementiaRiskProfile(BaseModel):
    """Structured risk profile extracted from clinical narrative"""

    age_years: Optional[int] = Field(None, ge=0, le=120, description="Age in years")
    sex: Optional[str] = Field(None, pattern="^(M|F|Male|Female)$", description="Sex")

    # Risk factors
    hypertension: Optional[bool] = Field(None, description="Hypertension diagnosis")
    diabetes: Optional[bool] = Field(None, description="Diabetes diagnosis")
    depression: Optional[bool] = Field(None, description="Depression diagnosis")
    afib: Optional[bool] = Field(None, description="atrial fibrillation")
    stroke: Optional[bool] = Field(None, description="Experienced stroke")
    brain_injury: Optional[bool] = Field(None, description="Brain injury diagnosis")
    obesity: Optional[bool] = Field(None, description="BMI >=30 or documented obesity")
    # hyperlipidemia: Optional[bool] = Field(None, description="Hyperlipidemia diagnosis")
    smoking: Optional[bool] = Field(None, description="Current or former smoker")
    alcohol: Optional[bool] = Field(None, description="Current or former alcoholic")

    # Complex fields
    # comorbidities: List[str] = Field(default_factory=list, description="Other medical conditions")
    # cognitive_symptoms: List[str] = Field(default_factory=list, description="Memory loss, confusion, aphasia, etc.")

    class Config:
        json_schema_extra = {
            "example": {
                "age_years": 75,
                "sex": "F",
                "hypertension": True,
                # "hyperlipidemia":True,
                "brain_injury":False,
                "diabetes": False,
                "depression": True,
                "smoking": False,
                "obesity": False,
                "afib": False,
                "stroke": False,
                "alcohol": True,
                # "comorbidities": ["osteoarthritis"],
                # "cognitive_symptoms": ["memory_loss"]
            }
        }

# Extraction function with Pydantic validation
def extract_features_pydantic(summary_text: str) -> Optional[DementiaRiskProfile]:
    """Extract features using LLM + Pydantic validation"""
    if not summary_text or len(summary_text) < 50:
        return None

    prompt = PROMPT_TEMPLATE.format(summary=summary_text[:1500])

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,  # Reduced for faster generation
            temperature=0.0,     # Greedy decoding for deterministic output
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True       # Speed up generation
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Extract JSON from response
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if not json_match:
        return None

    try:
        # Parse JSON
        data = json.loads(json_match.group())

        # Validate with Pydantic
        profile = DementiaRiskProfile(**data)
        return profile

    except (json.JSONDecodeError, ValidationError) as e:
        print(f"Validation failed: {e}")
        return None


# Run extraction with batch processing for Colab
llm_results = []
regex_results = []
batch_size = 10  # Process in small batches to manage VRAM

regex_extractor = RegexExtractor()
for i in tqdm(range(0, len(patients), batch_size)):
    batch = patients[i:i+batch_size]

    for patient in batch:
        summary = patient.get("patient", "")
        regex_extracted = regex_extractor.extract(summary)
        llm_extracted = extract_features_pydantic(summary)
        if i < 5:
          print("\n------\n")
          print(summary)
          print("\n------\n")
          print(llm_extracted)
          print("\n------\n")
          print(regex_extracted)

        regex_results.append(regex_extracted)
        if llm_extracted:
            llm_results.append({
                "patient_id": patient['patient_uid'],
                "extraction_success": True,
                "age_years": getattr(llm_extracted, "age_years"),
                "sex": getattr(llm_extracted, "sex"),
                "hypertension": getattr(llm_extracted, "hypertension"),
                # "hyperlipidemia": getattr(llm_extracted, "hyperlipidemia"),
                "brain_injury":getattr(llm_extracted, "brain_injury"),
                "diabetes": getattr(llm_extracted, "diabetes"),
                "depression": getattr(llm_extracted,"depression"),
                "smoking": getattr(llm_extracted,"smoking"),
                "obesity": getattr(llm_extracted,"obesity"),
                "afib": getattr(llm_extracted,"afib"),
                "stroke": getattr(llm_extracted,"stroke"),
                "alcohol": getattr(llm_extracted,"alcohol"),

            })

        else:
            llm_results.append({
                "patient_id": patient['patient_uid'],
                "extraction_success": False
            })

    # Clear cache periodically
    if i % 50 == 0:
        torch.cuda.empty_cache()

llm_results_df = pd.DataFrame(llm_results)
llm_results_df.to_csv("data/processed/llm_extracted_features.csv", index=False)

regex_results_df = pd.DataFrame(regex_results)
regex_results_df.to_csv("data/processed/regex_extracted_features.csv", index=False)

print(f"\nSuccess rate: {llm_results_df['extraction_success'].mean():.2%}")
print(f"Final VRAM usage: ~{torch.cuda.memory_allocated()/1e9:.1f} GB")

/tmp/ipykernel_2782/1915796721.py:135: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class DementiaRiskProfile(BaseModel):
  0%|          | 0/70 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



------

A 10-year old African-American female with Down syndrome was evaluated by our pulmonary service for history of chronic daily cough and recurrent pneumonias for eight and a half years duration. Cough was worse at night, in supine position and during exertion. Cough also worsened during viral respiratory tract infections. There was history of frequent vomiting of undigested food but not necessarily associated with the cough.
Patient was diagnosed with asthma exacerbations and pneumonia and treated as such several times in her lifetime. She had history of transient clinical improvement with antibiotics, bronchodilators and oral corticosteroids. Daily inhaled corticosteroids failed to completely control cough. Past medical history was significant for small ventricular septal defect and chronic constipation.
On examination, she was between 75th and 90th percentile for weight, and between 10th and 25th percentile for height. Chest examination was unremarkable. Chest roentgenograms s

  1%|▏         | 1/70 [02:03<2:22:31, 123.93s/it]

Validation failed: Extra data: line 14 column 1 (char 217)

------

A 64-year-old diabetic lady underwent pars plana vitrectomy in her left eye for a taut posterior hyaloid face due to proliferative diabetic retinopathy (PDR). Visual acuity (VA) at baseline had been 20/120. She had previously received panretinal laser photocoagulation (PRP) and the retinopathy had been stable, but there was localized extrafoveal tractional retinal detachment in the inferonasal quadrant. After vitrectomy, she was discharged in good condition, VA of 20/400 and mild vitreous hemorrhage (VH). One month postoperatively, the density of the VH increased and VA decreased to counting fingers (CF) (). The VH was non-clearing for three months but on echography, the retina was attached (). VH density decreased one month later and the patient received additional peripheral laser therapy. Six months postoperatively, she underwent uncomplicated phacoemulsification with intraocular lens (IOL) implantation due to sever

  3%|▎         | 2/70 [03:50<2:08:57, 113.79s/it]

...

Validation failed: Extra data: line 14 column 1 (char 217)
Validation failed: Expecting property name enclosed in double quotes: line 12 column 1 (char 192)
Validation failed: Extra data: line 14 column 1 (char 217)


 99%|█████████▊| 69/70 [2:16:57<01:59, 119.84s/it]

Validation failed: Extra data: line 14 column 1 (char 217)
Validation failed: Extra data: line 14 column 1 (char 217)
Validation failed: Extra data: line 14 column 1 (char 217)
Validation failed: Extra data: line 14 column 1 (char 217)


100%|██████████| 70/70 [2:18:45<00:00, 118.94s/it]


Success rate: 77.14%
Final VRAM usage: ~5.2 GB


In [ ]:
def compare_extractions(llm_results, regex_results):
    """
    Compare LLM vs regex outputs.

    Args:
        llm_results: list of dicts (your llm_results)
        regex_results: list of dicts (your regex_results)

    Returns:
        summary metrics + per-field accuracy
    """

    assert len(llm_results) == len(regex_results), "Mismatched lengths"

    fields = [
        "age_years", "sex",
        "hypertension", "brain_injury",
        "diabetes", "depression", "smoking", "obesity",
        "afib", "stroke", "alcohol"
    ]

    total = len(llm_results)
    field_correct = {f: 0 for f in fields}
    field_total = {f: 0 for f in fields}

    exact_match_count = 0

    for llm, regex in zip(llm_results, regex_results):

        # Skip failed LLM extractions
        if not llm.get("extraction_success", False):
            continue

        all_match = True

        for field in fields:
            llm_val = llm.get(field)
            regex_val = regex.get(field)

            # Only compare when both have values
            if llm_val is None or regex_val is None:
                continue

            field_total[field] += 1

            if llm_val == regex_val:
                field_correct[field] += 1
            else:
                all_match = False

        if all_match:
            exact_match_count += 1

    # Compute accuracies
    field_accuracy = {
        f: (field_correct[f] / field_total[f] if field_total[f] > 0 else None)
        for f in fields
    }

    overall_accuracy = exact_match_count / total if total > 0 else 0

    return {
        "total_samples": total,
        "exact_match_accuracy": overall_accuracy,
        "field_accuracy": field_accuracy,
        "field_counts": field_total
    }

metrics = compare_extractions(llm_results, regex_results)

print("\n=== OVERALL ===")
print(f"Exact match accuracy: {metrics['exact_match_accuracy']:.2%}")

print("\n=== PER FIELD ===")
for field, acc in metrics["field_accuracy"].items():
    if acc is not None:
        print(f"{field:15s}: {acc:.2%} (n={metrics['field_counts'][field]})")


=== OVERALL ===
Exact match accuracy: 46.71%

=== PER FIELD ===
age_years      : 98.13% (n=428)
sex            : 96.39% (n=527)
hypertension   : 92.45% (n=159)
brain_injury   : 55.77% (n=260)
diabetes       : 94.79% (n=192)
depression     : 92.86% (n=182)
smoking        : 82.52% (n=206)
obesity        : 85.96% (n=178)
afib           : 97.40% (n=192)
stroke         : 91.06% (n=179)
alcohol        : 87.70% (n=187)
